In [61]:
import json
import os
import re
import tempfile
from pathlib import Path

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [62]:
settings = json.loads(Path("settings.json").read_text(encoding="utf-8"))

update_news_data = bool(settings["update_news_data"])
news_end_date = pd.Timestamp(settings["news_end_date"]).date()
catalog_path = Path(settings["paths"]["news_catalog"])

sections = (
    "business",
    "money",
    "world",
    "politics",
    "us-news",
    "technology",
    "commentisfree",
    "news",
    "uk-news",
)
search_url = "https://content.guardianapis.com/search"
page_size = 200

In [63]:
def clean_text(value):
    if value is None:
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()


def make_session():
    retry = Retry(
        total=4,
        backoff_factor=1.0,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=("GET",),
    )
    adapter = HTTPAdapter(max_retries=retry)
    session = requests.Session()
    session.mount("https://", adapter)
    return session


def normalize_article(item, page):
    fields = item.get("fields") or {}
    body_text = clean_text(fields.get("bodyText"))
    return {
        "id": item.get("id", ""),
        "type": item.get("type", ""),
        "sectionId": item.get("sectionId", ""),
        "sectionName": item.get("sectionName", ""),
        "webPublicationDate": item.get("webPublicationDate", ""),
        "webTitle": item.get("webTitle", ""),
        "webUrl": item.get("webUrl", ""),
        "apiUrl": item.get("apiUrl", ""),
        "tags": json.dumps(item.get("tags", []), ensure_ascii=True),
        "references": json.dumps(item.get("references", []), ensure_ascii=True),
        "rights": json.dumps(item.get("rights", {}), ensure_ascii=True),
        "isHosted": item.get("isHosted", ""),
        "pillarId": item.get("pillarId", ""),
        "pillarName": item.get("pillarName", ""),
        "headline": fields.get("headline", ""),
        "trailText": "",
        "bodyText_200": "",
        "wordcount": "",
        "byline": "",
        "_fetch_page": page,
    }

In [64]:
def fetch_section(session, section, start_date, end_date, api_key, max_results=None):
    base_params = {
        "api-key": api_key,
        "from-date": start_date.isoformat(),
        "to-date": end_date.isoformat(),
        "section": section,
        "page-size": page_size,
        "q": "S&P 500 OR GSPC OR \"S&P\"",
        "order-by": "oldest",
        "show-fields": "headline",
        "show-tags": "all",
        "show-references": "all",
        "show-rights": "all",
    }

    rows = []
    expected_total = None
    expected_pages = None
    for page in range(1, 100000):
        params = {**base_params, "page": page}
        response = session.get(search_url, params=params, timeout=60)
        response.raise_for_status()
        payload = response.json()["response"]

        live_total = int(payload["total"])
        live_pages = int(payload["pages"])
        expected_total = live_total if expected_total is None else expected_total
        expected_pages = live_pages if expected_pages is None else expected_pages
        if live_total != expected_total or live_pages != expected_pages:
            raise RuntimeError(f"Guardian result count changed for {section}.")
        if int(payload["currentPage"]) != page:
            raise RuntimeError(f"Guardian page mismatch for {section}.")

        rows.extend(normalize_article(item, page) for item in payload["results"])
        if page >= expected_pages:
            break

    if len(rows) != expected_total:
        raise RuntimeError(f"Guardian row count mismatch for {section}.")
    return rows


def atomic_write_catalog(frame, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w",
            encoding="utf-8",
            dir=path.parent,
            prefix=f".{path.name}.",
            suffix=".tmp",
            delete=False,
        ) as handle:
            temporary_path = Path(handle.name)
            frame.to_csv(handle, index=False)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary_path, path)
    finally:
        if temporary_path is not None and temporary_path.exists():
            temporary_path.unlink()

In [65]:
catalog = pd.read_csv(catalog_path, low_memory=False)
# Normalize existing catalog publication dates
catalog["webPublicationDate"] = pd.to_datetime(
    catalog["webPublicationDate"],
    errors="coerce",
    utc=True,
)

# Desired full range from settings.json
desired_start = pd.Timestamp(settings.get("news_start_date")).date()
desired_end = pd.Timestamp(settings.get("news_end_date")).date()

# Determine existing coverage in catalog
if len(catalog) > 0:
    existing_pub = pd.to_datetime(catalog["webPublicationDate"], errors="coerce", utc=True)
    existing_min = existing_pub.min().date()
    existing_max = existing_pub.max().date()
else:
    existing_min = None
    existing_max = None

print(f"Catalog covers: {existing_min} -> {existing_max} (desired: {desired_start} -> {desired_end})")

# Decide which ranges to fetch
ranges_to_fetch = []
if update_news_data:
    # Force refresh the entire desired range
    ranges_to_fetch.append((desired_start, desired_end))
else:
    if existing_min is None:
        ranges_to_fetch.append((desired_start, desired_end))
    else:
        if existing_min > desired_start:
            start = desired_start
            end = (pd.Timestamp(existing_min) - pd.Timedelta(days=1)).date()
            if start <= end:
                ranges_to_fetch.append((start, end))
        if existing_max < desired_end:
            start = (pd.Timestamp(existing_max) + pd.Timedelta(days=1)).date()
            end = desired_end
            if start <= end:
                ranges_to_fetch.append((start, end))

# Helper to split a date range into year-long chunks
import time

def date_chunks(start_date, end_date, chunk_days=365):
    start_ts = pd.Timestamp(start_date)
    end_ts = pd.Timestamp(end_date)
    cur = start_ts
    while cur <= end_ts:
        chunk_end = cur + pd.Timedelta(days=chunk_days - 1)
        if chunk_end > end_ts:
            chunk_end = end_ts
        yield (cur.date(), chunk_end.date())
        cur = chunk_end + pd.Timedelta(days=1)

# Helper to iterate through each day in a date range
def daily_dates(start_date, end_date):
    cur = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)
    while cur <= end:
        yield cur.date()
        cur += pd.Timedelta(days=1)

# Fetch ranges in 1-year chunks, fetching all headlines for each day/section, save after each year
updated_flag = False
chunk_days = 365  # Process in 1-year chunks
pause_seconds = 0.1

if ranges_to_fetch:
    api_key = os.getenv("GUARDIAN_API_KEY", "").strip()
    if not api_key:
        raise RuntimeError("GUARDIAN_API_KEY is missing. Set it in the environment to fetch news.")
    session = make_session()
    for (rstart, rend) in ranges_to_fetch:
        # Split the requested range into 1-year chunks
        for (cstart, cend) in date_chunks(rstart, rend, chunk_days=chunk_days):
            print(f"Processing year chunk: {cstart} -> {cend}")
            year_rows = []
            
            # For each day in this year chunk, fetch all available headlines for each section
            for day in daily_dates(cstart, cend):
                for section in sections:
                    try:
                        section_rows = fetch_section(session, section, day, day, api_key)
                        year_rows.extend(section_rows)
                        time.sleep(pause_seconds)
                    except Exception as e:
                        print(f"Warning: failed fetching section {section} for {day}: {e}")

            # Save after processing the entire year chunk
            if year_rows:
                year_df = pd.DataFrame(year_rows)
                year_df["webPublicationDate"] = pd.to_datetime(
                    year_df["webPublicationDate"], errors="coerce", utc=True
                )
                catalog = pd.concat([catalog, year_df], ignore_index=True)
                catalog = catalog.drop_duplicates("id", keep="last")
                catalog = catalog.sort_values(["webPublicationDate", "id"])
                catalog_to_save = catalog.copy()
                catalog_to_save["webPublicationDate"] = catalog_to_save["webPublicationDate"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
                atomic_write_catalog(catalog_to_save, catalog_path)
                updated_flag = True
                print(f"Saved year chunk: {cstart} -> {cend} (rows added: {len(year_df)})")

# Final publication summary
publication_time = pd.to_datetime(
    catalog["webPublicationDate"],
    errors="coerce",
    utc=True,
)
summary = pd.Series(
    {
        "rows": len(catalog),
        "unique_articles": catalog["id"].nunique(),
        "dates": publication_time.dt.date.nunique(),
        "start": publication_time.min().date().isoformat() if not publication_time.isna().all() else None,
        "end": publication_time.max().date().isoformat() if not publication_time.isna().all() else None,
        "updated": bool(updated_flag),
    },
    name="news_catalog",
)

display(summary.to_frame())

Catalog covers: 2005-01-01 -> 2017-01-03 (desired: 2005-01-01 -> 2026-07-31)
Processing year chunk: 2017-01-04 -> 2018-01-03


KeyboardInterrupt: 

In [ ]:
# If we fetched updates, merge and save
if updates_all:
    updates = pd.DataFrame(updates_all)
    updates["webPublicationDate"] = pd.to_datetime(
        updates["webPublicationDate"],
        errors="coerce",
        utc=True,
    )
    catalog = pd.concat([catalog, updates], ignore_index=True)
    catalog = catalog.drop_duplicates("id", keep="last")
    catalog = catalog.sort_values(["webPublicationDate", "id"])
    catalog["webPublicationDate"] = catalog["webPublicationDate"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    atomic_write_catalog(catalog, catalog_path)
    updated_flag = True
else:
    updated_flag = False

# Final publication summary
publication_time = pd.to_datetime(
    catalog["webPublicationDate"],
    errors="coerce",
    utc=True,
)
summary = pd.Series(
    {
        "rows": len(catalog),
        "unique_articles": catalog["id"].nunique(),
        "dates": publication_time.dt.date.nunique(),
        "start": publication_time.min().date().isoformat() if not publication_time.isna().all() else None,
        "end": publication_time.max().date().isoformat() if not publication_time.isna().all() else None,
        "updated": bool(updated_flag),
    },
    name="news_catalog",
)

display(summary.to_frame())

,news_catalog
rows,300355
unique_articles,300355
dates,4375
start,2005-01-01
end,2017-01-03
updated,True


In [66]:
# ============================================================================
# S&P 500 (^GSPC) News Fetcher - Standalone Cell
# ============================================================================
# Fetch S&P 500 related news for a specific date range

import time

# Set your date range here
sp500_start_date = pd.Timestamp("2024-01-01").date()
sp500_end_date = pd.Timestamp("2024-01-05").date()

# S&P 500 related keywords
sp500_query = "S&P 500 OR GSPC OR \"S&P\""

# Fetch from business and money sections only
sp500_sections = ("business", "money")

sp500_results = []

for section in sp500_sections:
    print(f"Fetching {section} for {sp500_start_date} to {sp500_end_date}...")
    
    params = {
        "api-key": api_key,
        "from-date": sp500_start_date.isoformat(),
        "to-date": sp500_end_date.isoformat(),
        "section": section,
        "q": sp500_query,
        "page-size": page_size,
        "order-by": "oldest",
        "show-fields": "headline,trailText",
        "show-tags": "all",
    }
    
    page = 1
    total_fetched = 0
    
    while True:
        params["page"] = page
        try:
            response = session.get(search_url, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()
            
            if "response" not in data or "results" not in data["response"]:
                break
            
            results = data["response"]["results"]
            if not results:
                break
            
            for item in results:
                sp500_results.append({
                    "id": item.get("id"),
                    "section": item.get("sectionId"),
                    "date": item.get("webPublicationDate"),
                    "title": clean_text(item.get("webTitle", "")),
                    "url": item.get("webUrl"),
                    "headline": clean_text(item.get("fields", {}).get("headline", "")),
                    "trailText": clean_text(item.get("fields", {}).get("trailText", "")),
                })
            
            total_fetched += len(results)
            pages = data["response"].get("pages", 1)
            
            if page >= pages:
                break
            
            page += 1
            time.sleep(0.1)  # Rate limit
            
        except Exception as e:
            print(f"  Error on page {page}: {e}")
            break
    
    print(f"  Fetched {total_fetched} articles from {section}\n")
    time.sleep(0.5)

# Display results
sp500_df = pd.DataFrame(sp500_results)
print(f"Total S&P 500 related articles: {len(sp500_df)}")
print(f"Date range: {sp500_df['date'].min()} to {sp500_df['date'].max()}")
print("\nFirst 10 articles:")
print(sp500_df[["date", "section", "title"]].head(10))


Fetching business for 2024-01-01 to 2024-01-05...
  Fetched 46 articles from business

Fetching money for 2024-01-01 to 2024-01-05...
  Fetched 12 articles from money

Total S&P 500 related articles: 58
Date range: 2024-01-01T07:00:24Z to 2024-01-05T16:43:57Z

First 10 articles:
                   date   section  \
0  2024-01-01T07:00:24Z  business   
1  2024-01-01T11:03:11Z  business   
2  2024-01-01T13:00:31Z  business   
3  2024-01-01T13:49:56Z  business   
4  2024-01-02T07:00:54Z  business   
5  2024-01-02T07:00:55Z  business   
6  2024-01-02T08:00:55Z  business   
7  2024-01-02T09:17:46Z  business   
8  2024-01-02T09:24:45Z  business   
9  2024-01-02T12:37:40Z  business   

                                               title  
0  Retailers want rest of UK to follow Scotland’s...  
1  Big five oil companies to reward shareholders ...  
2  Experts advise: how can this tiny – and funky ...  
3  China’s many systemic problems dominate its ou...  
4  Footsie turns 40: FTSE 100’s rise,